# Day 1 통합 캡스톤 (심화) — 신호 배치 전처리·특징추출 파이프라인 최적화

> **CuPy 2일 집중 코스 — Day 1 마무리 통합 실습 (약 60~75분, 심화)**

Day 1의 핵심 기법을 **하나의 현실적 프로젝트**로 통합하고, **3단계로 점진 최적화**합니다.

| 단계 | 적용 기법 | 출처 |
|------|-----------|------|
| 표준화·밴드패스·특징 | ndarray·`cupy.fft` | 02·03 |
| 차원축소 PCA | `cupy.linalg.eigh` | 03 |
| **v1 최적화** | `@cupy.fuse`·`out=`·전송 최소화 | 03·05 |
| **v2 최적화** | 스트림 청크 오버랩 | 06 |
| 측정·검증 | `benchmark`·이벤트·`allclose` | 01·05·06 |

## 목표
- naive(v0) → 메모리최적(v1) → 스트림오버랩(v2)로 **단계별 가속**을 달성·측정한다.
- 모든 버전이 CPU 기준과 **정확히 일치**함을 검증한다.
- 이벤트 프로파일로 **병목 단계**를 찾는다.

In [ ]:
import numpy as np, cupy as cp, math, time
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare, allclose
from cupyx.profiler import time_range
print_env()

B, N = 512, 50_000      # 512개 신호, 길이 50k
K = 16                  # 스펙트럼 밴드 수(특징 차원)
LO, HI = 50, 5000       # 밴드패스 유지 구간(rfft 빈)
NCOMP = 3               # PCA 주성분 수
xpmod = lambda a: cp.get_array_module(a)

## Stage 0 — 데이터 생성 (GPU에서 직접)
검증을 위해 동일 데이터를 host에도 둡니다(실전에선 GPU 생성만).

In [ ]:
rng = np.random.default_rng(0)
t = np.linspace(0, 1, N, endpoint=False).astype(np.float32)
sig = (np.sin(2*np.pi*60*t) + 0.5*np.sin(2*np.pi*1200*t)).astype(np.float32)
X_np = (sig[None,:] + 0.8*rng.standard_normal((B,N)).astype(np.float32)).astype(np.float32)
X_cp = cp.asarray(X_np)
print('X:', X_cp.shape, X_cp.dtype)

## Stage 1 — 행별 표준화 (TODO)
각 신호(행)를 z-score로 표준화. `axis=1, keepdims=True` 브로드캐스팅(02).

In [ ]:
def standardize(X):
    # TODO: xp 선택 후 (X - 행평균)/(행표준편차 + 1e-6)
    raise NotImplementedError

<details><summary>💡 해답 보기</summary>

```python
def standardize(X):
    xp = xpmod(X)
    mu = X.mean(axis=1, keepdims=True); sd = X.std(axis=1, keepdims=True) + 1e-6
    return (X - mu) / sd
```
</details>

## Stage 2 — 밴드패스 FFT (TODO)
`rfft`(axis=1) 후 `[LO:HI]` 밴드 밖을 0으로, `irfft`로 복원(03 FFT).

In [ ]:
def bandpass(Z, lo=LO, hi=HI):
    # TODO: F=xp.fft.rfft(Z,axis=1); F[:,:lo]=0; F[:,hi:]=0; return xp.fft.irfft(F,n=Z.shape[1],axis=1)
    raise NotImplementedError

<details><summary>💡 해답 보기</summary>

```python
def bandpass(Z, lo=LO, hi=HI):
    xp = xpmod(Z)
    F = xp.fft.rfft(Z, axis=1)
    F[:, :lo] = 0; F[:, hi:] = 0
    return xp.fft.irfft(F, n=Z.shape[1], axis=1)
```
</details>

## Stage 3 — 비선형 + 밴드파워 특징 (TODO)
비선형 `G = tanh(Y)·exp(-Y²)`(원소연산, 나중에 `fuse` 대상) 후, `G`의 파워스펙트럼을 **K개 밴드로 합산**해 특징 `(B, K)` 생성.

In [ ]:
def nonlin(Y):
    return cp.tanh(Y) * cp.exp(-Y*Y) if xpmod(Y) is cp else np.tanh(Y)*np.exp(-Y*Y)

def band_power(G, K=K):
    # TODO: P=|rfft(G,axis=1)|^2 -> 앞쪽을 K*(M//K)로 잘라 (B,K,-1) reshape 후 axis=2 합
    raise NotImplementedError

<details><summary>💡 해답 보기</summary>

```python
def band_power(G, K=K):
    xp = xpmod(G)
    P = xp.abs(xp.fft.rfft(G, axis=1))**2     # (B, M)
    M = P.shape[1]; w = M // K
    return P[:, :K*w].reshape(G.shape[0], K, w).sum(axis=2)   # (B, K)
```
</details>

## Stage 4 — PCA 차원축소 (제공)
특징 `(B,K)`를 `eigh`로 PCA(03). 고유벡터 부호 모호성 때문에 **고유값**으로 검증합니다.

In [ ]:
def pca_values(F, k=NCOMP):
    xp = xpmod(F)
    Fc = F - F.mean(axis=0, keepdims=True)
    Cov = (Fc.T @ Fc) / (F.shape[0]-1)
    w = xp.linalg.eigvalsh(Cov)
    return xp.sort(w)[::-1][:k]      # 상위 k 고유값

## 조립 — v0 (naive) & 정확성 검증
단계를 그대로 이어 붙인 기준 버전입니다. CPU/GPU 결과(고유값)가 일치하는지 확인하세요.

In [ ]:
def pipeline_v0(X):
    Z = standardize(X)
    Y = bandpass(Z)
    G = nonlin(Y)
    F = band_power(G)
    return pca_values(F)

# (해답 구현 후 실행)
# ref = pipeline_v0(X_np)
# out = cp.asnumpy(pipeline_v0(X_cp))
# allclose(ref, out, rtol=2e-2, atol=2e-2, name='pipeline v0 (PCA eigvals)')

## 최적화 v1 — `@cupy.fuse` + `out=` + 전송 최소화 (TODO)
비선형 단계를 **융합 커널**로 만들고(temporary 제거, 03·05), 파이프라인이 끝까지 **GPU에 머물게** 하세요.
`pca_values`까지 GPU에서 수행하고 마지막에 한 번만 host로 가져옵니다.

In [ ]:
# TODO: @cp.fuse() 로 nonlin_fused 정의, pipeline_v1(X_cp) 구성
# @cp.fuse()
# def nonlin_fused(Y): ...
# def pipeline_v1(X): ...
raise NotImplementedError

<details><summary>💡 해답 보기</summary>

```python
@cp.fuse()
def nonlin_fused(Y):
    return cp.tanh(Y) * cp.exp(-Y*Y)

def pipeline_v1(X):                 # X는 GPU 배열 가정
    Z = standardize(X)
    Y = bandpass(Z)
    G = nonlin_fused(Y)            # 융합 커널 (중간배열 감소)
    F = band_power(G)
    return pca_values(F)

_ = nonlin_fused(X_cp[:2])         # 워밍업(첫 호출 컴파일)
out1 = cp.asnumpy(pipeline_v1(X_cp))
allclose(cp.asnumpy(pipeline_v0(X_cp)), out1, rtol=1e-3, atol=1e-3, name='v1==v0')
```
</details>

## 최적화 v2 — 스트림 청크 오버랩 (TODO)
배치 `B`를 행 방향 청크로 나눠 **여러 스트림**에서 전처리(표준화→밴드패스→비선형→밴드파워)를 겹쳐 수행한 뒤,
모은 특징으로 PCA를 합니다(06). 청크 처리는 서로 독립이라 오버랩에 적합합니다.

In [ ]:
def features_chunked(X, nstreams=3, chunk=128):
    # TODO: X를 chunk 행씩 나눠 streams[i%nstreams]에서 band_power(nonlin_fused(bandpass(standardize(.))))
    #       각 청크 특징을 모아 (B,K)로 결합 후 반환
    raise NotImplementedError

def pipeline_v2(X, nstreams=3, chunk=128):
    return pca_values(features_chunked(X, nstreams, chunk))
# out2 = cp.asnumpy(pipeline_v2(X_cp)); allclose(cp.asnumpy(pipeline_v1(X_cp)), out2, rtol=1e-3, atol=1e-3, name='v2==v1')

<details><summary>💡 해답 보기</summary>

```python
def features_chunked(X, nstreams=3, chunk=128):
    B0 = X.shape[0]
    feats = cp.empty((B0, K), dtype=cp.float32)
    streams = [cp.cuda.Stream(non_blocking=True) for _ in range(nstreams)]
    for i, s0 in enumerate(range(0, B0, chunk)):
        with streams[i % nstreams]:
            xb = X[s0:s0+chunk]
            g  = nonlin_fused(bandpass(standardize(xb)))
            feats[s0:s0+chunk] = band_power(g)
    for st in streams: st.synchronize()
    return feats

def pipeline_v2(X, nstreams=3, chunk=128):
    return pca_values(features_chunked(X, nstreams, chunk))

out2 = cp.asnumpy(pipeline_v2(X_cp))
allclose(cp.asnumpy(pipeline_v1(X_cp)), out2, rtol=1e-3, atol=1e-3, name='v2==v1')
```
</details>

## 성능 비교 — v0(CPU) vs v0/v1/v2(GPU)
각 버전을 측정하고 **단계별 가속**을 확인하세요. 성능 목표: v2(GPU)가 v0(CPU) 대비 큰 폭의 speedup.

In [ ]:
# (구현 후 실행)
# print_bench(bench(lambda: pipeline_v0(X_np),  n_repeat=3, name='v0 CPU'))
# print_bench(bench(lambda: pipeline_v0(X_cp),  n_repeat=5, name='v0 GPU naive'))
# print_bench(bench(lambda: pipeline_v1(X_cp),  n_repeat=5, name='v1 fuse+mem'))
# print_bench(bench(lambda: pipeline_v2(X_cp),  n_repeat=5, name='v2 streams'))

## 병목 프로파일 — 이벤트로 단계별 시간
어느 단계(표준화/FFT/비선형/특징/PCA)가 지배적인지 이벤트로 측정해 보세요(06).

In [ ]:
def profile_stages(X):
    evs = [cp.cuda.Event() for _ in range(6)]
    evs[0].record()
    Z = standardize(X);            evs[1].record()
    Y = bandpass(Z);               evs[2].record()
    G = nonlin_fused(Y);           evs[3].record()
    F = band_power(G);             evs[4].record()
    _ = pca_values(F);             evs[5].record()
    evs[5].synchronize()
    names = ['standardize','bandpass','nonlin','band_power','pca']
    for i,nm in enumerate(names):
        print(f'{nm:>11}: {cp.cuda.get_elapsed_time(evs[i],evs[i+1]):7.3f} ms')

# profile_stages(X_cp)   # 해답 구현 후 실행

## 도전 과제 (택1+)

<details><summary>아이디어 펼치기</summary>

- **밴드패스+비선형까지 한 패스로**: `bandpass` 결과에 `out=` 버퍼를 재사용해 메모리 트래픽을 더 줄여 보세요.
- **pinned + `blocking=False`**: host에서 데이터를 받는 시나리오로 바꿔 전송과 연산을 진짜로 겹쳐 보세요(06).
- **CUDA Graph**: 동일 크기 배치를 반복 처리한다면 v2를 그래프로 캡처해 런치 오버헤드를 줄여 보세요(06 심화).
- **특징 확장**: 밴드파워 외에 첨도/엔트로피(`cupyx.scipy.stats`)를 추가해 특징을 늘리고 PCA 효과를 보세요(04).
- **dtype**: float64로 바꾸면 정확도/속도가 어떻게 변하나요?
</details>

## 제출물 체크리스트
- [ ] Stage 1~3 (`standardize`/`bandpass`/`band_power`) 구현
- [ ] v0 CPU/GPU 고유값 `allclose` 통과
- [ ] v1(`fuse`+메모리)·v2(스트림) 구현 및 v0와 일치 검증
- [ ] v0(CPU) → v2(GPU) **speedup 표**
- [ ] 이벤트 프로파일로 병목 단계 식별
- [ ] (선택) 도전 과제 1개 + 개선 효과 3줄

수고하셨습니다! **Day 2**는 사용자 정의 CUDA 커널(`07_cupy_kernels`)로 이어집니다.